In [1]:
import torch
import numpy as np
import spock_reg_model
from tqdm.notebook import tqdm
from utils import assert_equal
from types import SimpleNamespace
from sklearn.metrics import roc_auc_score
import pickle

/home/sca63/.conda/envs/planet_eqs/lib/python3.7/site-packages/juliacall/__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  "torch was imported before juliacall. This may cause a segfault. "


Detected Jupyter notebook. Loading juliacall extension. Set `PYSR_AUTOLOAD_EXTENSIONS=no` to disable.


## Model configs & data loading (run once, then switch with `load_model`)

In [3]:
# load the cache
with open('model_cache.pkl', 'rb') as f:
    cache2 = pickle.load(f)
    # don't ever need the model
    _cache = {k: (v[0], v[1], v[2], None) for k, v in cache2.items()}

In [4]:
from calc_rmse import get_dataloader

MODELS = {
    'nn_24880': dict(version=24880),
    'nn_28114': dict(version=28114),
    # 'nn_12370': dict(version=12370),
    'pysr_24880_11003_26': dict(version=24880, pysr_version=11003, pysr_model_selection=26),
    'pysr_28114_41564_35': dict(version=28114, pysr_version=41564, pysr_model_selection=35),
    # 'pysr_12318_93102_29': dict(version=12318, pysr_version=93102, pysr_model_selection=29),
    # 'pysr_28114_93890_43': dict(version=28114, pysr_version=93890, pysr_model_selection=43),
}

# Cache: {(model_key, dataset): (xs, truths, preds, model)}
# _cache = {}

def load_model(model_key, dataset='val'):
    """Load model + data. Caches results so re-runs are instant."""
    global xs, truths, preds, model, current_model, current_dataset

    cache_key = (model_key, dataset)
    if cache_key in _cache:
        xs, truths, preds, model = _cache[cache_key]
        current_model = model_key
        current_dataset = dataset
        # print(f'[cached] {model_key} on {dataset}: {len(truths)} samples')
        return

    cfg = MODELS[model_key]
    dataloader = get_dataloader(dataset, petit=False)

    if 'pysr_version' in cfg:
        m = spock_reg_model.load_with_pysr_f2(
            version=cfg['version'],
            pysr_version=cfg['pysr_version'],
            pysr_model_selection=cfg['pysr_model_selection'],
        )
    else:
        m = spock_reg_model.load(version=cfg['version'])
    m.eval()
    m.cuda()

    def predict_fn(x):
        return m(x.cuda(), noisy_val=False, deterministic=True).cpu().detach()

    _xs, _truths, _preds = [], [], []
    for x, y in tqdm(dataloader):
        _xs.append(x)
        _truths.append(y)
        _preds.append(predict_fn(x))

    xs = torch.cat(_xs, dim=0)
    truths = torch.cat(_truths, dim=0)
    preds = torch.cat(_preds, dim=0)
    model = m
    current_model = model_key
    current_dataset = dataset
    assert_equal(truths.shape, preds.shape)

    _cache[cache_key] = (xs, truths, preds, model)
    # print(f'{model_key} on {dataset}: {len(truths)} samples')
    # print(f'  xs mean={xs.mean().item():.4f}  truths mean={truths.mean().item():.4f}  preds mean={preds[:, 0].mean().item():.4f}')

# let's just preload all the models + data
for model_key in MODELS:
    for dataset in ['val', 'test', 'random', 'train']:
        load_model(model_key, dataset=dataset)

print('All models and data loaded.')

All models and data loaded.


In [ ]:
# cache2 = {k: (v[0], v[1], v[2]) for k, v in _cache.items()}

# import pickle
# # save the cache, so we can load it later
# with open('model_cache.pkl', 'wb') as f:
#     pickle.dump(cache2, f)

In [5]:
def safe_log_erf(x):
    base_mask = x < -1
    value_giving_zero = torch.zeros_like(x, device=x.device)
    x_under = torch.where(base_mask, x, value_giving_zero)
    x_over = torch.where(~base_mask, x, value_giving_zero)

    f_under = lambda x: (
         0.485660082730562*x + 0.643278438654541*torch.exp(x) +
         0.00200084619923262*x**3 - 0.643250926022749 - 0.955350621183745*x**2
    )
    f_over = lambda x: torch.log(1.0+torch.erf(x))

    return f_under(x_under) + f_over(x_over)


def lossfnc(testy, y):
    # y: [B, 2] batch of ground truth means. each input system has two
    #  simulations, one with a small initial perturbation, so the two means
    #  are samples from the distribution of instability times for that
    #  initial system
    # so we just sum over the loss for both of them.
    mu = testy[:, [0]]
    std = testy[:, [1]]
    # std = torch.ones_like(std)

    var = std**2
    t_greater_9 = y >= 9

    regression_loss = -(y - mu)**2/(2*var)
    regression_loss += -torch.log(std)

    regression_loss += -safe_log_erf(
                (mu - 4)/(torch.sqrt(2*var))
            )

    # classifier_loss = safe_log_erf(
    #             (mu - 9)/(torch.sqrt(2*var))
    #     )
    classifier_loss = (
        safe_log_erf((mu - 9) / torch.sqrt(2*var)) -
        safe_log_erf((mu - 4) / torch.sqrt(2*var))
    )

    safe_regression_loss = torch.where(
            ~torch.isfinite(regression_loss),
            -torch.ones_like(regression_loss)*100,
            regression_loss)
    safe_classifier_loss = torch.where(
            ~torch.isfinite(classifier_loss),
            -torch.ones_like(classifier_loss)*100,
            classifier_loss)

    total_loss = (
        safe_regression_loss * (~t_greater_9) +
        safe_classifier_loss * ( t_greater_9)
    )
    # return -total_loss.sum(1)

    total_regression = safe_regression_loss * (~t_greater_9)
    total_classifier = safe_classifier_loss * ( t_greater_9)
    total_regression = -(total_regression.sum() / len(y)).item()
    total_classifier = -(total_classifier.sum() / len(y)).item()
    avg_std = std.mean().item()

    total_loss = total_regression + total_classifier

    return total_loss, total_regression, total_classifier, avg_std

## Run loss + metrics across all models

In [6]:
from sklearn.metrics import roc_auc_score

SPLITS = ['train', 'val', 'test', 'random']

def single_class_acc(truths, preds):
    thresholds = np.sort(np.unique(preds))[::-1]  # sweep high -> low
    stable_truths = truths >= 9

    def accuracy_at_threshold(t):
        predicted_stable = preds >= t
        acc = np.mean(predicted_stable == stable_truths)
        return acc
    scores = [accuracy_at_threshold(t) for t in thresholds]
    best_threshold = thresholds[np.argmax(scores)]
    return best_threshold, np.max(scores)

def calculate_metrics_from_cache():
    """Calculate RMSE/acc/AUC/bias from the cached preds and truths."""
    p = np.clip(preds[:, 0].numpy(), 4, 9)
    t = np.average(truths.numpy(), axis=1)

    unstable = t < 9
    rmse = np.sqrt(np.mean((t[unstable] - p[unstable])**2))
    acc = np.mean((p >= 9) == (t >= 9))
    stable_ixs = t >= 9
    stable_only_acc = np.mean((p[stable_ixs] >= 9) == (t[stable_ixs] >= 9))
    auc = roc_auc_score(t >= 9, p)
    # stable_best_acc = single_class_acc(t, p)
    bias = np.mean(p - t)
    return dict(rmse=rmse, acc=acc, auc=auc, bias=bias, stable_only_acc=stable_only_acc)

for name in MODELS:
    print(f'=== {name} ===')
    for split in SPLITS:
        load_model(name, dataset=split)
        loss, reg, classifier, std_mean = lossfnc(preds, truths)
        # loss2 = model._lossfnc(preds, truths).sum() / len(truths)
        m = calculate_metrics_from_cache()
        # print(f'  [{split:6s}] loss={loss:.4f}  '
        #       f'RMSE={m["rmse"]:.4f}  Acc={m["acc"]:.4f}  Stable-only Acc={m["stable_only_acc"]:.4f}  reg={reg:.4f}  cls={classifier:.4f}  std_mean={std_mean:.4f}')
        print(f'  [{split:6s}] loss={loss:.4f}  '
              f'RMSE={m["rmse"]:.4f}  Acc={m["acc"]:.4f} reg_loss={reg:.4f}  cls_loss={classifier:.4f}  std_mean={std_mean:.4f}')
    print()

=== nn_24880 ===
  [train ] loss=1.8862  RMSE=1.0554  Acc=0.8787 reg_loss=1.5593  cls_loss=0.3269  std_mean=1.4693
  [val   ] loss=1.9673  RMSE=1.1033  Acc=0.8714 reg_loss=1.6266  cls_loss=0.3406  std_mean=1.4662
  [test  ] loss=2.0001  RMSE=1.1196  Acc=0.8748 reg_loss=1.6422  cls_loss=0.3579  std_mean=1.4694
  [random] loss=1.7079  RMSE=1.2595  Acc=0.8373 reg_loss=1.0637  cls_loss=0.6442  std_mean=1.3855

=== nn_28114 ===
  [train ] loss=1.9365  RMSE=1.0876  Acc=0.8723 reg_loss=1.5984  cls_loss=0.3381  std_mean=1.6352
  [val   ] loss=2.0739  RMSE=1.1611  Acc=0.8603 reg_loss=1.7065  cls_loss=0.3674  std_mean=1.6324
  [test  ] loss=2.1049  RMSE=1.1628  Acc=0.8623 reg_loss=1.7143  cls_loss=0.3906  std_mean=1.6340
  [random] loss=1.7725  RMSE=1.2608  Acc=0.8267 reg_loss=1.0823  cls_loss=0.6902  std_mean=1.4732

=== pysr_24880_11003_26 ===
  [train ] loss=2.6273  RMSE=1.3312  Acc=0.8084 reg_loss=1.9599  cls_loss=0.6674  std_mean=1.4829
  [val   ] loss=2.6237  RMSE=1.3409  Acc=0.8100 reg_lo

In [ ]:
from calc_rmse import get_args, calc_metrics

In [ ]:
args = get_args()
args.version =24880
args.pysr_version = 11003
args.eval_type = 'pysr'

for model_selection in